# LiPAD — Corrosion Detection Training

Train a **YOLO segmentation** corrosion detector with Roboflow-aligned settings.

## Quick start (Google Colab)

1. **Runtime → Change runtime type → T4 GPU** (or better).
2. Upload or clone `LIPAD_YOLO_TRAINING` into `/content/LIPAD_YOLO_TRAINING`.
3. Put raw images/labels under `corrosion_detection/datasets_raw/` **or** directly in `corrosion_detection/datasets/images/{train,val}` with matching `labels/{train,val}`.
4. Run all cells top to bottom.
5. Best weights are saved under `corrosion_detection/runs/<model>/corrosion_<model>_seg/weights/best.pt`.

## Configuration applied

| Setting | Value |
|---|---|
| Epochs | 100 |
| Learning rate | 0.01 |
| Optimizer | SGD |
| Image size | 640×640 (stretch) |
| Auto-orient | Yes |
| Contrast | Adaptive equalization (CLAHE) |
| Classes | fair, poor, severe (2 remapped, 3 dropped) |
| Aug copies (train) | 3 per source image (offline photometric) |
| Flips | Horizontal + vertical |
| Rotation | ±15° |
| Exposure | ±25% |
| Blur | up to 2.5 px |
| Noise | up to 10% of pixels |

Edit defaults in `shared/corrosion_config.py` if your Roboflow export uses different class ids.

In [1]:
%pip install torch torchvision
%pip install ultralytics pyyaml albumentations opencv-python-headless pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.4 MB/s eta 0:00:0000:01


In [1]:
# @title 1. Environment setup
import sys
from pathlib import Path

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content
    # After cloning/uploading, point here if needed:
    # %cd /content/LIPAD_YOLO_TRAINING
else:
    ROOT = Path(r'C:/Users/Admin/PROJECT_LIPAD/Corrosion/LIPAD_YOLO_TRAINING')
    if (Path.cwd() / 'shared').exists():
        ROOT = Path.cwd()
    elif (Path.cwd().parent / 'shared').exists():
        ROOT = Path.cwd().parent
    %cd {ROOT}
    sys.path.insert(0, str(ROOT))
    print('Repo root:', ROOT)


Mounted at /content/drive
/content


In [14]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [16]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/LIPAD_TRAINING_VERSION2")

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

print("Google Drive training directory:", DRIVE_ROOT)
print("Exists:", DRIVE_ROOT.exists())

Google Drive training directory: /content/drive/MyDrive/LIPAD_TRAINING_VERSION2
Exists: True


In [17]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/LIPAD_YOLO_TRAINING/LIPAD_YOLO_TRAINING"
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Shared exists:", (PROJECT_ROOT / "shared").exists())
print("Config exists:", (PROJECT_ROOT / "shared" / "corrosion_config.py").exists())

Project root: /content/LIPAD_YOLO_TRAINING/LIPAD_YOLO_TRAINING
Shared exists: True
Config exists: True


In [21]:
%pip install -r /content/LIPAD_YOLO_TRAINING/LIPAD_YOLO_TRAINING/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.1 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of notebook to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 30.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.6/120.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.8/109.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 89.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 139.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 135.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.3/915.3 kB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 6.5 MB/s eta 0:00:00
  Attempting uninstall: jupyter-client
 

In [22]:
import inspect
from shared.trainer import train_corrosion

print(inspect.signature(train_corrosion))

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
(family: 'ModelFamily', *, config: "'CorrosionTrainConfig | None'" = None, batch: 'int | None' = None, device: 'str | int | None' = None, workers: 'int | None' = None, project_name: 'str | None' = None, run_name: 'str | None' = None, resume: 'bool' = False) -> 'Path'


In [23]:
import inspect
from shared.trainer import train_corrosion

print(inspect.getsource(train_corrosion))

def train_corrosion(
    family: ModelFamily,
    *,
    config: "CorrosionTrainConfig | None" = None,
    batch: int | None = None,
    device: str | int | None = None,
    workers: int | None = None,
    project_name: str | None = None,
    run_name: str | None = None,
    resume: bool = False,
) -> Path:
    """Train corrosion segmentation with Roboflow-aligned hyperparameters."""
    if corrosion_train_kwargs is None or write_corrosion_dataset_yaml is None:
        raise ImportError("shared.corrosion_config is required for train_corrosion()")

    task_dir = "corrosion_detection"
    data_yaml = write_corrosion_dataset_yaml()
    spec = get_model_spec("corrosion", family)

    if device is None:
        device = 0 if torch.cuda.is_available() else "cpu"
    if workers is None:
        workers = 2 if os.name == "nt" else 4
        if device == "cpu":
            workers = 0

    project = project_name or str(runs_dir(task_dir) / family)
    name = run_name or f"corrosion_{family}_se

In [24]:
import inspect
from shared.trainer import train_corrosion

source = inspect.getsource(train_corrosion)

for i, line in enumerate(source.splitlines(), 1):
    if "project" in line.lower() or "run_name" in line.lower() or "model.train" in line.lower():
        print(f"{i:3}: {line}")

  8:     project_name: str | None = None,
  9:     run_name: str | None = None,
 27:     project = project_name or str(runs_dir(task_dir) / family)
 28:     name = run_name or f"corrosion_{family}_seg"
 42:         project=project,
 48:     model.train(data=str(data_yaml), **train_kwargs)
 50:     best = Path(project) / name / "weights" / "best.pt"


In [25]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/LIPAD_TRAINING")

CORROSION_RUNS = DRIVE_ROOT / "corrosion" / "runs"
CORROSION_RUNS.mkdir(parents=True, exist_ok=True)

print("Saving corrosion training to:")
print(CORROSION_RUNS)

Saving corrosion training to:
/content/drive/MyDrive/LIPAD_TRAINING/corrosion/runs


In [18]:
CORROSION_DRIVE = DRIVE_ROOT / "corrosion"
CRACK_DRIVE = DRIVE_ROOT / "crack"

for directory in [
    CORROSION_DRIVE / "weights",
    CORROSION_DRIVE / "runs",
    CORROSION_DRIVE / "results",
    CRACK_DRIVE / "weights",
    CRACK_DRIVE / "runs",
    CRACK_DRIVE / "results",
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Training directories created.")

Training directories created.


In [2]:
!git clone https://github.com/sarieljandaniel-svg/Corrosion.git /content/LIPAD_YOLO_TRAINING

Cloning into '/content/LIPAD_YOLO_TRAINING'...
remote: Enumerating objects: 16211, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 16211 (delta 11), reused 10 (delta 10), pack-reused 16186 (from 1)
Receiving objects: 100% (16211/16211), 1.58 GiB | 31.19 MiB/s, done.
Resolving deltas: 100% (1169/1169), done.
Updating files: 100% (16098/16098), done.


In [9]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/LIPAD_YOLO_TRAINING/LIPAD_YOLO_TRAINING")

print("Project root:", PROJECT_ROOT)
print("Exists:", PROJECT_ROOT.exists())
print("Shared exists:", (PROJECT_ROOT / "shared").exists())
print("Config exists:", (PROJECT_ROOT / "shared" / "corrosion_config.py").exists())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

Project root: /content/LIPAD_YOLO_TRAINING/LIPAD_YOLO_TRAINING
Exists: True
Shared exists: True
Config exists: True


In [13]:
!ls -la /content/LIPAD_YOLO_TRAINING/LIPAD_YOLO_TRAINING

total 48
drwxr-xr-x 5 root root 4096 Aug 11 05:47 .
drwxr-xr-x 5 root root 4096 Aug 11 05:47 ..
drwxr-xr-x 7 root root 4096 Aug 11 05:59 corrosion_detection
drwxr-xr-x 4 root root 4096 Aug 11 05:47 crack_detection
-rw-r--r-- 1 root root   64 Aug 11 05:47 .gitattributes
-rw-r--r-- 1 root root  353 Aug 11 05:47 .gitignore
-rw-r--r-- 1 root root  246 Aug 11 05:47 LIPAD_YOLO_TRAINING.code-workspace
-rw-r--r-- 1 root root 1462 Aug 11 05:47 one_epoch.py
-rw-r--r-- 1 root root 2037 Aug 11 05:47 README.md
-rw-r--r-- 1 root root  238 Aug 11 05:47 requirements.txt
-rw-r--r-- 1 root root  449 Aug 11 05:47 setup.ps1
drwxr-xr-x 3 root root 4096 Aug 11 05:58 shared


In [ ]:
# @title 2. Review training configuration

from shared.corrosion_config import (
    config_summary,
    CorrosionPreprocessConfig,
    CorrosionTrainConfig,
)

print(config_summary())

print("\nPreprocess config:", CorrosionPreprocessConfig())
print("Train config     :", CorrosionTrainConfig())

Base training
  epochs=100, lr0=0.01, optimizer=SGD
Preprocessing
  auto_orient=True
  resize=640x640 (stretch)
  clahe=True (adaptive equalization)
  class_remap=4 ids, drop=[2, 5, 6]
  offline_augment_copies=3
Augmentations (training + optional offline preprocess)
  flips: horizontal + vertical
  rotation: ±15.0°
  exposure: ±25%
  blur: up to 2.5px
  noise: up to 10% of pixels
  classes: {0: 'fair', 1: 'poor', 2: 'severe'}

Preprocess config: CorrosionPreprocessConfig(image_size=640, apply_auto_orient=True, apply_clahe=True, clahe_clip_limit=2.0, clahe_tile_grid_size=(8, 8), resize_mode='stretch', offline_augment_copies=3, horizontal_flip=True, vertical_flip=True, rotation_degrees=15.0, exposure_fraction=0.25, blur_max_pixels=2.5, noise_max_pixel_fraction=0.1)
Train config     : CorrosionTrainConfig(epochs=100, lr0=0.01, optimizer='SGD', imgsz=640, batch=16, horizontal_flip_prob=0.5, vertical_flip_prob=0.5, rotation_degrees=15.0, exposure_fraction=0.25, blur_max_pixels=2.5, noise_ma

In [4]:
!find /content/LIPAD_YOLO_TRAINING -maxdepth 5 -type f -name "corrosion_config.py"

/content/LIPAD_YOLO_TRAINING/LIPAD_YOLO_TRAINING/shared/corrosion_config.py


In [5]:
!find /content/LIPAD_YOLO_TRAINING -maxdepth 4 -type d -name "shared"

/content/LIPAD_YOLO_TRAINING/LIPAD_YOLO_TRAINING/shared


In [6]:
!ls -la /content/LIPAD_YOLO_TRAINING

total 456
drwxr-xr-x  5 root root   4096 Aug 11 05:47 .
drwxr-xr-x  1 root root   4096 Aug 11 05:46 ..
drwxr-xr-x  8 root root   4096 Aug 11 05:47 .git
drwxr-xr-x  5 root root   4096 Aug 11 05:47 LIPAD_YOLO_TRAINING
drwxr-xr-x 10 root root   4096 Aug 11 05:47 PROJECT_LIPAD
-rw-r--r--  1 root root 443933 Aug 11 05:47 Project_LiPAD.ipynb


In [11]:
# @title 3. Prepare dataset folders
from shared.paths import ensure_dataset_layout, datasets_dir, task_root

ensure_dataset_layout('corrosion_detection')
raw_root = task_root('corrosion_detection') / 'datasets_raw'
for sub in ('images/train', 'images/val', 'labels/train', 'labels/val'):
    (raw_root / sub).mkdir(parents=True, exist_ok=True)

print('Processed dataset :', datasets_dir('corrosion_detection'))
print('Optional raw input  :', raw_root)
print('Expected YOLO layout:')
print('  datasets_raw/images/train  +  datasets_raw/labels/train')
print('  datasets_raw/images/val    +  datasets_raw/labels/val')
print('Classes: fair (0), poor (1), severe (2)')

Processed dataset : /content/LIPAD_YOLO_TRAINING/LIPAD_YOLO_TRAINING/corrosion_detection/datasets
Optional raw input  : /content/LIPAD_YOLO_TRAINING/LIPAD_YOLO_TRAINING/corrosion_detection/datasets_raw
Expected YOLO layout:
  datasets_raw/images/train  +  datasets_raw/labels/train
  datasets_raw/images/val    +  datasets_raw/labels/val
Classes: fair (0), poor (1), severe (2)


In [12]:
# @title 4. Preprocess (auto-orient, stretch 640, CLAHE, class remap, 3x train copies)
from shared.preprocess_corrosion import preprocess_corrosion_dataset
from shared.paths import datasets_dir, task_root

RAW_DIR = task_root('corrosion_detection') / 'datasets_raw'
USE_RAW = any((RAW_DIR / 'images' / 'train').glob('*'))

yaml_path = preprocess_corrosion_dataset(
    raw_dir=RAW_DIR if USE_RAW else None,
    clear_existing=True,
)
print('Wrote dataset yaml:', yaml_path)
if USE_RAW:
    print('Raw folder used:', RAW_DIR)
else:
    print('In-place reprocess of:', datasets_dir('corrosion_detection'))

Wrote dataset yaml: /content/LIPAD_YOLO_TRAINING/LIPAD_YOLO_TRAINING/corrosion_detection/dataset.yaml
In-place reprocess of: /content/LIPAD_YOLO_TRAINING/LIPAD_YOLO_TRAINING/corrosion_detection/datasets


In [ ]:
# @title 5. Train YOLOv8 (configured)
from shared.trainer import train_corrosion

best_v8 = train_corrosion(
    "yolov8",
    batch=8,
    project_name=str(CORROSION_RUNS),
    run_name="yolov8_train",
)

print("Done:", best_v8)

In [ ]:
# @title 6. Train YOLOv11 (configured)
from shared.trainer import train_corrosion

best_v11 = train_corrosion('yolov11', batch=16 if IN_COLAB else 8)
print('Done:', best_v11)

In [ ]:
# @title 7. Train YOLOv12 (configured)
from shared.trainer import train_corrosion

best_v12 = train_corrosion('yolov12', batch=16 if IN_COLAB else 8)
print('Done:', best_v12)

## Step-by-step training guide

### A. Prepare your data

**Option 1 — Roboflow export (recommended)**

1. Export from Roboflow as **YOLOv8 Segmentation**.
2. Match these Roboflow settings if possible:
   - Auto-Orient: on
   - Resize: Stretch 640×640
   - Auto-Adjust Contrast: Adaptive Equalization
   - Modify Classes: 2 remapped, 3 dropped
   - Augmentations: 3 outputs, flips, ±15° rotation, ±25% exposure, blur ≤2.5 px, noise ≤10%
3. Unzip into `corrosion_detection/datasets_raw/` with this layout:

```
datasets_raw/
  images/train/
  images/val/
  labels/train/
  labels/val/
```

**Option 2 — Already in YOLO layout**

Place files directly under `corrosion_detection/datasets/images/{train,val}` and `labels/{train,val}`. Cell 4 will reprocess them in place.

### B. Adjust class mapping (if needed)

Open `shared/corrosion_config.py` and edit:

- `CORROSION_CLASS_REMAP` — map old class ids to fair/poor/severe
- `CORROSION_DROP_CLASS_IDS` — ids to ignore

Default output classes: **fair (0), poor (1), severe (2)**.

### C. Run the notebook

| Cell | Action |
|---|---|
| 1 | Install deps + set repo path |
| 2 | Print active configuration |
| 3 | Create folder structure |
| 4 | Preprocess images (orient, stretch, CLAHE, remap, 3 train copies) |
| 5–7 | Train YOLOv8 / v11 / v12 (pick one or run all) |

### D. After training

- Weights: `corrosion_detection/runs/<model>/corrosion_<model>_seg/weights/best.pt`
- Metrics/plots: same run folder
- Download `best.pt` from Colab: Files panel → right-click → Download

### E. Notes

- **3 outputs per example:** offline copies use photometric aug (exposure/blur/noise). Flips and rotation are applied during YOLO training with correct mask transforms.
- **Batch size:** lower to `8` or `4` if Colab runs out of GPU memory.
- **Resume training:** call `train_corrosion('yolov8', resume=True)`.